# Linly-Dubbing Kaggle WebUI (Fixed)

This notebook is optimized for running **Linly-Dubbing** on Kaggle with **Dual T4 GPUs**.

### 🚀 Features
- **Streamlined deployment**: 5 simple steps with comprehensive checks
- **Environment detection**: Verify system state before installation
- **No Conda overhead**: Utilizes Kaggle's native Python environment
- **Dual T4 optimized**: Leverages parallel GPU capabilities
- **Smart caching**: Reduces redundant downloads
- **Robust error handling**: Better dependency installation flow

### 📋 Execution Guide
0. **Step 0**: Detect and verify runtime environment
1. **Step 1**: Clone repository and check GPU availability
2. **Step 2**: Install all dependencies (system + Python packages)
3. **Step 3**: Download required AI models
4. **Step 4**: Launch the WebUI

### ⚙️ Kaggle Setup Requirements
- Enable **GPU T4 x2** in Settings → Accelerator
- Enable **Internet** in Settings → Internet

---

In [ ]:
# ============================================================================
# Step 0: 环境检测 (Environment Detection)
# ============================================================================

import os
import sys
import platform
import subprocess
import shutil

print("=" * 60)
print("🔍 Kaggle 运行环境检测")
print("=" * 60)

# Python 信息
print("\n📊 Python 环境:")
print(f"   Python 版本: {sys.version.split()[0]}")
print(f"   Python 路径: {sys.executable}")
print(f"   平台: {platform.platform()}")

# CUDA 和 PyTorch 信息
print("\n🎮 CUDA 环境:")
try:
    import torch
    print(f"   PyTorch 版本: {torch.__version__}")
    print(f"   CUDA 可用: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"   CUDA 版本: {torch.version.cuda}")
        try:
            cudnn_version = torch.backends.cudnn.version()
            print(f"   cuDNN 版本: {cudnn_version}")
        except RuntimeError as e:
            print(f"   cuDNN 版本: 无法获取 (版本不兼容)")
except ImportError:
    print("   ⚠️ PyTorch 未安装")

# 磁盘空间
print("\n💾 磁盘空间:")
try:
    stat = shutil.disk_usage('/kaggle/working')
    total_gb = stat.total / (1024**3)
    used_gb = stat.used / (1024**3)
    free_gb = stat.free / (1024**3)
    print(f"   总空间: {total_gb:.1f} GB")
    print(f"   已使用: {used_gb:.1f} GB")
    print(f"   可用: {free_gb:.1f} GB")
    if free_gb < 5:
        print("   ⚠️ 警告: 可用空间不足 5 GB")
except Exception as e:
    print(f"   ⚠️ 无法获取磁盘信息: {e}")

# 系统工具
print("\n🔧 系统工具:")
tools = ['git', 'gcc', 'g++', 'make', 'wget', 'curl', 'ffmpeg']
for tool in tools:
    if shutil.which(tool):
        print(f"   ✅ {tool}: 已安装")
    else:
        print(f"   ❌ {tool}: 未找到")

# 已安装的关键包
print("\n📦 预装 Python 包 (关键):")
key_packages = ['torch', 'torchvision', 'numpy', 'pandas', 'matplotlib', 'pip']
for pkg in key_packages:
    try:
        module = __import__(pkg)
        version = getattr(module, '__version__', '未知版本')
        print(f"   ✅ {pkg}: {version}")
    except ImportError:
        print(f"   ❌ {pkg}: 未安装")

print("\n" + "=" * 60)
print("✅ 环境检测完成!")
print("=" * 60)

In [ ]:
# ============================================================================
# Step 1: Clone Repository and Check GPU
# ============================================================================

import os
import torch

# Check GPU availability
print("=" * 60)
print("🔍 GPU Detection")
print("=" * 60)
if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    print(f"✅ Found {gpu_count} GPU(s):")
    for i in range(gpu_count):
        print(f"   - GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"     Memory: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB")
else:
    print("❌ No GPU detected! Please enable GPU in Kaggle settings.")
    print("   Settings → Accelerator → GPU T4 x2")

print("\n" + "=" * 60)
print("📦 Cloning Repository")
print("=" * 60)

# Clone repository if not exists
if not os.path.exists('/kaggle/working/Linly-Dubbing'):
    print("Cloning Linly-Dubbing repository...")
    !cd /kaggle/working && git clone https://github.com/Kedreamix/Linly-Dubbing.git --depth 1
    print("✅ Repository cloned successfully")
else:
    print("Repository already exists, pulling latest changes...")
    !cd /kaggle/working/Linly-Dubbing && git pull

# Change to project directory
%cd /kaggle/working/Linly-Dubbing

# Initialize submodules
print("\nInitializing submodules...")
!git submodule update --init --recursive

print("\n✅ Step 1 Complete!")
print("=" * 60)

In [ ]:
# ============================================================================
# Step 2: Install Dependencies (Enhanced for Kaggle)
# ============================================================================

import subprocess
import sys
import os

print("=" * 60)
print("📦 Installing System Dependencies")
print("=" * 60)

# 1. Install system tools required for building pynini and other C++ extensions
print("Installing build-essential and libfst-dev...")
!apt-get update -qq && apt-get install -y -qq build-essential libfst-dev ffmpeg

print("\n" + "=" * 60)
print("🐍 Installing Python Dependencies")
print("=" * 60)

# 2. Patch requirements.txt (Fix numpy compatibility and remove conflicting pynini versions)
print("Applying compatibility patches to requirements.txt...")
!sed -i 's/numpy==1.26.3/numpy<2.0.0/g' requirements.txt
!sed -i '/pynini/d' requirements.txt # Remove pynini from requirements to handle separately

# 3. Patch TTS for Python 3.12 compatibility (Kaggle default)
if os.path.exists('submodules/TTS/setup.py'):
    print("Patching TTS for Python 3.12+ compatibility...")
    !sed -i 's/Version(python_version) >= Version(\"3.12\"):/Version(python_version) >= Version(\"3.13\"):/g' submodules/TTS/setup.py
    !sed -i 's/python_requires=\">=3.9.0, <3.12\",/python_requires=\">=3.9.0, <3.13\",/g' submodules/TTS/setup.py

# 4. Install pynini from source/wheel safely
print("\nInstalling pynini...")
!pip install -q pynini==2.1.5 || pip install -q pynini

# 5. Install basic requirements (using current PyTorch env)
print("\nInstalling core project requirements...")
!pip install -q -r requirements.txt

# 6. CRITICAL: Install submodules in EDITABLE mode to fix 'ModuleNotFoundError'
# This ensures that 'demucs.api' and other internal paths are correctly resolved
print("\nInstalling project submodules in editable mode...")
submodules = ['submodules/demucs', 'submodules/whisper', 'submodules/whisperX', 'submodules/TTS']
for sm in submodules:
    if os.path.exists(sm):
        print(f"   - Installing {sm}...")
        !pip install -q -e {sm}
    else:
        print(f"   ⚠️ Module {sm} not found, skipping...")

# 7. Install any remaining missing critical tools
print("\nEnsuring critical tools are installed...")
!pip install -q loguru yt-dlp gradio==4.44.1

# ============================================================================
# Final Verification
# ============================================================================
print("\n" + "=" * 60)
print("✅ Verifying Critical Modules")
print("=" * 60)

test_modules = [
    ('demucs.api', 'Separator'), 
    ('gradio', 'Interface'),
    ('torch', '__version__'),
    ('whisper', 'load_model'),
    ('whisperx', 'load_align_model')
]

all_passed = True
for mod_path, attr in test_modules:
    try:
        mod = __import__(mod_path, fromlist=[attr])
        print(f"   ✅ {mod_path}: OK")
    except Exception as e:
        print(f"   ❌ {mod_path}: FAIL ({e})")
        all_passed = False

if all_passed:
    print("\n🎉 Step 2 Complete! All critical dependencies are ready.")
else:
    print("\n⚠️ Some dependencies failed to verify. Check logs above.")
print("=" * 60)

In [ ]:
# ============================================================================
# Step 3: Download AI Models
# ============================================================================

print("=" * 60)
print("🤖 Downloading AI Models")
print("=" * 60)
print("⏱️ This may take 10-15 minutes depending on network speed.")
print("📊 Total download size: ~15 GB")
print("")

# Create model directories
!mkdir -p models/ASR/whisper

# Download wav2vec2 model for WhisperX
import os
wav2vec_path = 'models/ASR/whisper/wav2vec2_fairseq_base_ls960_asr_ls960.pth'

if os.path.exists(wav2vec_path):
    print("✅ wav2vec2 model already exists, skipping download.")
else:
    print("📥 Downloading wav2vec2 model (360 MB)...")
    !wget -nc https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960_asr_ls960.pth \
        -O {wav2vec_path}
    print("✅ wav2vec2 model downloaded")

# Download HuggingFace models using the project's download script
print("\n📥 Downloading HuggingFace models (XTTS-v2, Qwen1.5, faster-whisper)...")
!python scripts/huggingface_download.py

print("\n✅ Step 3 Complete!")
print("=" * 60)

In [ ]:
# ============================================================================
# Step 4: Launch WebUI
# ============================================================================

import os

print("=" * 60)
print("🚀 Launching Linly-Dubbing WebUI")
print("=" * 60)

# Set Matplotlib backend for headless environment
os.environ['MPLBACKEND'] = 'Agg'

# Create .env file if not exists
if not os.path.exists('.env'):
    print("Creating .env configuration file...")
    !cp env.example .env
    print("✅ .env file created")

print("\n🌐 Starting Gradio WebUI with Public Link...")
print("🔗 Click the 'xxxxx.gradio.live' link when it appears.")
print("=" * 60)
print("")

# Launch WebUI
# Note: webui.py should have share=True in launch() for Kaggle to work
!python webui.py